# Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets

import torch.utils.data as data
from torch.utils.data import random_split

from PIL import Image

import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score

import numpy as np

from tqdm.notebook import tqdm

import os
import random

### Set device and seed

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # set device
print(device)

# Load Dataset

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = datasets.EuroSAT(root='./data', transform=transform, download=True)

## EDA

In [ ]:
# Distribution of classes

class_to_idx = dataset.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}

class_counts = {}
for _, label in tqdm(dataset):
    label = idx_to_class[label]

    if label in class_counts:
        class_counts[label] += 1
    else:
        class_counts[label] = 1



print("Class distribution:")
for k, v in class_counts.items():
    print(f"{k}: {v}")

In [ ]:
labels = class_counts.keys()
counts = class_counts.values()

fig, ax = plt.subplots(figsize=(10, 5))
ax.pie(counts, labels=labels, autopct="%1.1f%%")
plt.show()

## Splitting

In [ ]:
# 70-15-15 split of dataset into train, val and test sets using torch random split
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])


## Data Augmentation

In [ ]:
class customDataset(data.Dataset):
    """
    Custom dataset class to apply augmentations to the dataset.
    """
    def __init__(self, dataset, transform: transforms.Compose):
        self.dataset = dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx) :
        x, y = self.dataset[idx]

        # Apply transformations to the image
        if self.transform:
            x = self.transform(x)
        
        return x, y

In [ ]:
# Augmentations to apply: flipping, rotation, random crop and resize
transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.RandomResizedCrop(64, scale=(0.8, 1.0)),
])

train_dataset_aug = customDataset(train_dataset, transform=transform_aug)
val_dataset_aug = customDataset(val_dataset, transform=transform_aug)
test_dataset_enc = customDataset(test_dataset, transform=None)

# Sizes of the splits
print(f"Train dataset size: {len(train_dataset_aug)}")
print(f"Validation dataset size: {len(val_dataset_aug)}")
print(f"Test dataset size: {len(test_dataset_enc)}")

train_loader = data.DataLoader(train_dataset_aug, batch_size=128, shuffle=True)
val_loader = data.DataLoader(val_dataset_aug, batch_size=128, shuffle=False)
test_loader = data.DataLoader(test_dataset_enc, batch_size=1, shuffle=False)

# Training

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # A shortcut connection to match the dimensions of the input and output
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out
    
class ResNet(nn.Module):
    def __init__(self, num_classes=10):
        super(ResNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(64, 128, stride=2)
        self.layer2 = self._make_layer(128, 256, stride=2)
        self.layer3 = self._make_layer(256, 512, stride=2)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.conv2 = nn.Conv2d(512, num_classes, kernel_size=1)

    def _make_layer(self, in_channels, out_channels, stride):
        layers = []
        layers.append(ResidualBlock(in_channels, out_channels, stride))
        layers.append(ResidualBlock(out_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.gap(out)
        out = self.conv2(out)
        out = torch.flatten(out, 1) # flatten the output
        return out

In [ ]:
model = ResNet(num_classes=10).to(device)
print(model)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [ ]:
num_epochs = 20
train_losses = []
val_losses = []
model=model.to(device)

pbar = tqdm(range(num_epochs), desc="Training")
for epoch in pbar:
    model.train()
    running_loss = 0.0
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} Train", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    train_losses.append(running_loss / len(train_loader))

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} Val", leave=False):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
    val_losses.append(val_loss / len(val_loader))

    pbar.set_postfix(train_loss=f"{train_losses[-1]:.4f}", val_loss=f"{val_losses[-1]:.4f}")

# Plot the training and validation losses
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()